# SKEX on Kaggle

Turn the Accelerator on before Run. **GPU T4 x2 is the right choice** on this account. Internet **On**. The notebook uses only the first GPU, so a second T4 is not an error. A session with no GPU cannot load the models.

Add Data -> **`skex-datasets`**. On this account Kaggle mounts it at `/kaggle/input/datasets/umardrazbhatti/skex-datasets`. The files are:

- `/kaggle/input/datasets/umardrazbhatti/skex-datasets/processed/domain_a/{train,dev,test}.jsonl` — what the runner loads
- `/kaggle/input/datasets/umardrazbhatti/skex-datasets/sciriff/4096/` — SciRIFF parquet
- `/kaggle/input/datasets/umardrazbhatti/skex-datasets/scier/` — SciER JSONL
- `/kaggle/input/datasets/umardrazbhatti/skex-datasets/scierc/extracted/processed_data/json/` — SciERC JSONL
- `/kaggle/input/datasets/umardrazbhatti/skex-datasets/cord/text/` — CORD receipt text

This notebook clones `https://github.com/umardrazbhatti-work/skex`, checks those paths, and copies Domain A into the clone. It does not download a 7B model and it does not install Unsloth.

The smoke cell writes `/kaggle/working/skex-output.zip`. The last cell skips the sealed Qwen dev cells, scores the 99 Qwen test papers, then runs Llama 3.2 1B and 3B on dev and test. Prompt-JSON versus Outlines. Add a Kaggle secret named `HF_TOKEN` before Run. The token must be a read token from the Hugging Face account that accepted the Llama 3.2 license. T4 x2 is fine. Budget about 3–4 hours. The runner rewrites `skex-output.zip` during the run. Download it from the Output tab into a new Results folder. This notebook does not fine-tune.


In [ ]:
import os
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    listing = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    print(listing)
    gpus = [line for line in listing.splitlines() if line.strip()]
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print(f"{len(gpus)} GPU(s) visible to nvidia-smi. This process will use only GPU 0.")
    if gpus and "T4" not in listing:
        print("WARNING: the visible GPU is not named T4.")
else:
    print("No NVIDIA GPU visible. The smoke cells below do not need one.")


In [ ]:
import subprocess
import sys
from pathlib import Path

REMOTE = "https://github.com/umardrazbhatti-work/skex.git"

def run(cmd, cwd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd))

if Path("/kaggle/working").is_dir():
    work = Path("/kaggle/working")
    repo = work / "skex"
    if not (repo / ".git").exists():
        run(["git", "clone", REMOTE], work)
    else:
        run(["git", "pull", "--ff-only"], repo)
elif (Path.cwd() / "src" / "skex").is_dir():
    repo = Path.cwd()
    print(f"Already inside {repo}. Not cloning.")
else:
    raise SystemExit("Run this notebook on Kaggle, or from the skex repo root.")

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], repo)
print("repo", repo)


In [ ]:
import json
import os
import shutil
from pathlib import Path

DATA = Path("/kaggle/input/datasets/umardrazbhatti/skex-datasets")

PATHS = {
    "domain_a": DATA / "processed" / "domain_a",
    "sciriff": DATA / "sciriff" / "4096",
    "scier_llm": DATA / "scier" / "LLM",
    "scier_plm": DATA / "scier" / "PLM",
    "scierc": DATA / "scierc" / "extracted" / "processed_data" / "json",
    "cord": DATA / "cord" / "text",
}
REQUIRED = [
    PATHS["domain_a"] / "train.jsonl",
    PATHS["domain_a"] / "dev.jsonl",
    PATHS["domain_a"] / "test.jsonl",
    PATHS["sciriff"] / "train-00000-of-00001.parquet",
    PATHS["sciriff"] / "validation-00000-of-00001.parquet",
    PATHS["sciriff"] / "test-00000-of-00001.parquet",
    PATHS["scier_llm"] / "train.jsonl",
    PATHS["scier_llm"] / "dev.jsonl",
    PATHS["scier_llm"] / "test.jsonl",
    PATHS["scier_plm"] / "train.jsonl",
    PATHS["scierc"] / "train.json",
    PATHS["scierc"] / "dev.json",
    PATHS["scierc"] / "test.json",
    PATHS["cord"] / "train.jsonl",
    PATHS["cord"] / "dev.jsonl",
    PATHS["cord"] / "test.jsonl",
]

if not DATA.is_dir():
    attached = sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").is_dir() else []
    raise SystemExit(
        "Dataset not found at /kaggle/input/datasets/umardrazbhatti/skex-datasets.\n"
        "Add Data -> skex-datasets on this notebook, then run this cell again.\n"
        f"Attached inputs: {attached or 'none'}"
    )

missing = [str(path) for path in REQUIRED if not path.is_file()]
if missing:
    raise SystemExit("Attached dataset is missing files:\n" + "\n".join(missing))

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()
dest = repo / "data" / "processed" / "domain_a"
dest.mkdir(parents=True, exist_ok=True)
for name in ("train.jsonl", "dev.jsonl", "test.jsonl"):
    shutil.copy2(PATHS["domain_a"] / name, dest / name)
    n = sum(1 for line in (dest / name).open(encoding="utf-8") if line.strip())
    print(f"domain_a {name}: {n} rows <- {PATHS['domain_a'] / name}")

for label, folder in PATHS.items():
    print(f"{label}: {folder} exists={folder.is_dir()}")

registered = {key: str(path) for key, path in PATHS.items()}
registered["domain_a_repo"] = str(dest)
(repo / "data_paths.json").write_text(json.dumps(registered, indent=2), encoding="utf-8")
os.environ["SKEX_DATA_ROOT"] = str(DATA)
print("registered", json.dumps(registered, indent=2))
print("CORD text is registered but not copied into data/processed/domain_b. That converter is not written yet.")


In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()
failed = None
try:
    subprocess.check_call([sys.executable, "-m", "pytest", "-q"], cwd=str(repo))
    for _ in range(2):
        subprocess.check_call(
            [sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/00_smoke.yaml"],
            cwd=str(repo),
        )
    subprocess.check_call([sys.executable, "-m", "skex.experiments.status"], cwd=str(repo))
except Exception as exc:
    failed = exc
    print("RUN FAILED:", exc)

zip_path = Path("/kaggle/working/skex-output.zip") if Path("/kaggle/working").is_dir() else repo / "skex-output.zip"
include = []
runs = repo / "outputs" / "runs"
if runs.is_dir():
    include.extend(path for path in runs.rglob("*") if path.is_file())
for relative in ("experiments/registry.jsonl", "experiments/failures.jsonl", "data_paths.json"):
    path = repo / relative
    if path.is_file():
        include.append(path)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    note = "smoke failed\n" + str(failed) if failed else "smoke finished\n"
    archive.writestr("RUN.txt", note)
    for path in include:
        archive.write(path, path.relative_to(repo).as_posix())
print(f"Wrote {zip_path} with {len(include)} files. Download it from the Kaggle Output tab and put it in the Results folder next to the log.")
if failed:
    raise failed
print("Smoke is done. The next cell needs HF_TOKEN, then Qwen test and Llama dev plus test.")


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()

def pack():
    zip_path = Path("/kaggle/working/skex-output.zip") if Path("/kaggle/working").is_dir() else repo / "skex-output.zip"
    include = []
    runs = repo / "outputs" / "runs"
    if runs.is_dir():
        include.extend(path for path in runs.rglob("*") if path.is_file())
    for relative in ("experiments/registry.jsonl", "experiments/failures.jsonl", "data_paths.json"):
        path = repo / relative
        if path.is_file():
            include.append(path)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        note = "test plan failed during " + stage + "\n" + str(failed) if failed else "test plan finished\n"
        archive.writestr("RUN.txt", note)
        for path in include:
            archive.write(path, path.relative_to(repo).as_posix())
    print(f"Wrote {zip_path} with {len(include)} files. Download it from the Output tab into the Results folder.")

failed = None
stage = "not started"
try:
    gpu_list = ""
    if shutil.which("nvidia-smi"):
        gpu_list = subprocess.check_output(["nvidia-smi", "-L"], text=True)
        print(gpu_list)
    if "GPU" not in gpu_list:
        raise SystemExit(
            "This Kaggle session has no GPU, so the models cannot load. "
            "Turn the Accelerator on. GPU T4 x2 is accepted. Leave Internet On, then Run again."
        )
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "bitsandbytes", "outlines", "pydantic", "sentencepiece"], cwd=str(repo))
    import torch
    if not torch.cuda.is_available():
        raise SystemExit("nvidia-smi sees a GPU but torch cannot use CUDA. Restart the session with the Accelerator turned on.")
    print("torch device count", torch.cuda.device_count(), "name", torch.cuda.get_device_name(0))
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        print("HF token is set. Not printing it.")
    else:
        raise SystemExit(
            "Llama 3.2 is gated. Hugging Face has accepted the license, and this run includes those models. "
            "Create a read token on that Hugging Face account, then on Kaggle open Settings, Secrets, and add a secret labeled HF_TOKEN. "
            "Do not paste the token into the notebook. Run this cell again after the secret is saved."
        )
    child_env = os.environ.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["CUDA_VISIBLE_DEVICES"] = "0"
    child_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    stage = "dev skip"
    print("Sealed Qwen dev plan must skip all four cells.")
    subprocess.check_call([sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/01_tax_zeroshot.yaml", "--retry-failed"], cwd=str(repo), env=child_env)
    stage = "qwen test"
    print("Qwen test: 1.5B then 3B, prompt-JSON then Outlines, 99 papers.")
    subprocess.check_call([sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/01b_tax_zeroshot_test.yaml", "--retry-failed"], cwd=str(repo), env=child_env)
    stage = "llama dev"
    print("Llama dev: 3.2 1B then 3B, prompt-JSON then Outlines, 50 papers.")
    subprocess.check_call([sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/01c_tax_zeroshot_llama_dev.yaml", "--retry-failed"], cwd=str(repo), env=child_env)
    stage = "llama test"
    print("Llama test: 3.2 1B then 3B, prompt-JSON then Outlines, 99 papers.")
    subprocess.check_call([sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/01d_tax_zeroshot_llama_test.yaml", "--retry-failed"], cwd=str(repo), env=child_env)
    subprocess.check_call([sys.executable, "-m", "skex.experiments.status"], cwd=str(repo))
    stage = "done"
except Exception as exc:
    failed = exc
    print("TEST PLAN FAILED during", stage, exc)
finally:
    pack()
if failed:
    raise failed
print("Test plan finished. Do not start QLoRA until this zip is in the Results folder.")
